# Week 5 L2 — Data mining: descriptive analysis — solution

**Data mining** is finding useful patterns in data. Its first task, **summarisation**, describes a dataset with a few numbers: mean, minimum, maximum and standard deviation. This notebook applies them, in pandas and in MongoDB, first to the male and female data and then to the combined appointments file from Week 4.


You need `community_health_activity_with_population.csv`, `community_health_activity_clean.csv` and `area_population_clean.csv` in the same folder as this notebook.

## Set up

### Install the packages

In [ ]:
%pip install pymongo python-dotenv

### Settings

Your connection string and database name are kept in a file called `.env`, in the same folder as this notebook, so they are not written into the notebook itself:

```
MONGO_URI=paste the connection string from Canvas here
DB_NAME=MOD008909_your_student_id
```

In [ ]:
# ---- CONFIG ----
import os
from dotenv import load_dotenv

load_dotenv()   # reads the .env file in the notebook's folder

MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("DB_NAME")

if not MONGO_URI or not DB_NAME:
    print("MONGO_URI or DB_NAME not found: check that .env is in the same folder as this notebook")
else:
    print("Settings loaded for database:", DB_NAME)

### Your files

The CSV files sit in the same folder as this notebook.

In [ ]:
folder_path = "./"

print(os.listdir(folder_path))

### Connect

The male and female collections were loaded in Lecture 1 and are still in your database.

In [ ]:
import pandas as pd
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

male = db["sexDensitiesMale"]
female = db["sexDensitiesFemale"]

print("sexDensitiesMale :", male.count_documents({}), "documents")
print("sexDensitiesFemale :", female.count_documents({}), "documents")

Expected: 439 in each. If you see 0, run the loading cell from Lecture 1 first.

### A function to load the other files

The same function as Lecture 1, with one line added: pandas marks a missing value as NaN, and MongoDB's word for missing is `null` (`None` in Python), so the function swaps one for the other before inserting.

In [ ]:
def load_csv_to_collection(file_name, collection_name):
    df = pd.read_csv(folder_path + file_name)
    df = df.astype(object).where(df.notna(), None)   # NaN -> None, stored as null
    collection = db[collection_name]
    collection.drop()
    collection.insert_many(df.to_dict("records"))
    print(collection_name, ":", collection.count_documents({}), "documents")
    return collection

---
# Part A — Concepts

## 1. Data mining task: summarisation

Summarisation is the first **data mining** task: describing a dataset compactly before anything else is done with it.

### Four measures

A table with millions of rows cannot be read row by row. A few numbers can describe it: where the values **centre**, and how far they **spread**. They also expose errors, such as an impossible minimum or maximum.

| Measure | What it tells you | pandas | MongoDB |
|---|---|---|---|
| Mean | the typical value: total ÷ count | `.mean()` | `$avg` |
| Minimum | the smallest value | `.min()` | `$min` |
| Maximum | the largest value | `.max()` | `$max` |
| Standard deviation | how far values typically sit from the mean | `.std()` | `$stdDevSamp` |

**Range** = maximum − minimum: the full spread, but set by just two values. `.describe()` gives count, mean, standard deviation, minimum, quartiles and maximum in one go.

## 2. Summarising the male and female data

### In pandas

Bring the documents into pandas with `pd.DataFrame(list(...find()))`. The files mix countries and regions with local authorities, so filter to local authorities first.

In [ ]:
local_authorities = {"CODE": {"$regex": "^(E06|E07|E08|E09|W06|S12|N09)"}}

la_male = pd.DataFrame(list(male.find(local_authorities, {"_id": 0, "CODE": 1, "Location": 1, "AllAges": 1})))

print("Local authorities:", len(la_male))
print("Mean:              ", round(la_male["AllAges"].mean(), 1))
print("Minimum:           ", la_male["AllAges"].min())
print("Maximum:           ", la_male["AllAges"].max())
print("Standard deviation:", round(la_male["AllAges"].std(), 1))

A typical local authority has about 81,000 males, but the standard deviation (about 55,600) is large: local authorities differ hugely in size, from the Isles of Scilly to Birmingham.

### In MongoDB: aggregation

`find()` returns **documents**. Often the question needs an **answer** instead: a total, a count, an average for each group. Calculating such answers is **summarisation**, a data mining task, and MongoDB does it with `aggregate()`.

`aggregate()` takes a **pipeline**: a list of stages. Documents flow through the stages in order.

| Stage | What it does | Like |
|---|---|---|
| `$match` | keep only documents that pass a filter | the filter in `find()` |
| `$group` | combine documents into groups and calculate per group | pandas `.groupby()` |
| `$sort` | put the results in order | `.sort()` |
| `$limit` | keep the first n results | `.limit()` |

Inside `$group`, `_id` says what to group by (`None` means one group: everything), and each other line is a calculation: `$sum`, `$avg`, `$min`, `$max`. `"$AllAges"`, with a `$` in front, means "the value of the field `AllAges`".

In [ ]:
# Total males in the London boroughs, and how many boroughs
pipeline = [
    {"$match": {"CODE": {"$regex": "^E09"}}},
    {"$group": {"_id": None, "total_males": {"$sum": "$AllAges"}, "boroughs": {"$sum": 1}}},
]
for result in male.aggregate(pipeline):
    print(result)

`{"$sum": "$AllAges"}` adds up a field. `{"$sum": 1}` adds 1 for each document: a count.

The four measures from section 1 are `$avg`, `$min`, `$max` and `$stdDevSamp`. The same summary as pandas above, calculated inside MongoDB:

In [ ]:
pipeline = [
    {"$match": local_authorities},
    {"$group": {
        "_id": None,
        "mean": {"$avg": "$AllAges"},
        "minimum": {"$min": "$AllAges"},
        "maximum": {"$max": "$AllAges"},
        "std_dev": {"$stdDevSamp": "$AllAges"},
    }},
]
print(list(male.aggregate(pipeline)))

The same numbers as pandas: `$stdDevSamp` and pandas `.std()` both calculate the **sample** standard deviation.

### One result per group

Put a field in `_id` and you get one result per value. The first letter of the area code gives the nation (E, W, S, N), and `$substrCP` takes it: `{"$substrCP": ["$CODE", 0, 1]}` means "from `CODE`, starting at position 0, take 1 character".

In [ ]:
pipeline = [
    {"$match": local_authorities},
    {"$group": {
        "_id": {"$substrCP": ["$CODE", 0, 1]},
        "areas": {"$sum": 1},
        "total_males": {"$sum": "$AllAges"},
        "average_males": {"$avg": "$AllAges"},
    }},
    {"$sort": {"total_males": -1}},
]
pd.DataFrame(list(male.aggregate(pipeline))).round(1)

> **Question.** Northern Ireland has the fewest males in total but the highest average per area. How can both be true?

**Answer.** 

### `$sort` and `$limit` in a pipeline

The largest local authorities, the same answer as `find().sort().limit()`. A pipeline can do it too, and it can go on to further stages that `find()` cannot.

In [ ]:
pipeline = [
    {"$match": local_authorities},
    {"$sort": {"AllAges": -1}},
    {"$limit": 3},
    {"$project": {"_id": 0, "Location": 1, "AllAges": 1}},
]
for result in male.aggregate(pipeline):
    print(result)

`$project` chooses the fields to show, like the projection in `find()`.

### Variation in real data

Males per female in each of the 439 areas, for each age band: bring both collections into pandas, match them on `CODE`, and divide.

In [ ]:
AGE_BANDS = ["AllAges", "<18", "19-25", "26-30", "31-35", "35-45", "45-60", ">61"]

male_df = pd.DataFrame(list(male.find({}, {"_id": 0})))
female_df = pd.DataFrame(list(female.find({}, {"_id": 0})))
both = pd.merge(male_df, female_df, on=["CODE", "Location"], suffixes=("_male", "_female"))

ratios = pd.DataFrame({"Location": both["Location"]})
for band in AGE_BANDS:
    ratios[band] = both[band + "_male"] / both[band + "_female"]

ratios[AGE_BANDS].agg(["mean", "min", "max", "std"]).T.round(3)

The means are all close to 1, except over 61, where women outnumber men. The **standard deviation** shows which band varies most from place to place.

In [ ]:
projection_ratio = ["Location", "19-25"]
print(ratios.sort_values("19-25", ascending=False)[projection_ratio].head(3))
print(ratios.sort_values("19-25")[projection_ratio].head(3))

> **Question.** Which age band has the largest standard deviation? Richmondshire, in North Yorkshire, has about 257 men for every 100 women aged 19–25. What could explain it?

**Answer.** 

## 3. Summarising the Week 4 combined data

### A worked example

In Week 4 you merged the appointment data with the population data into `community_health_activity_with_population.csv`. This cell loads it into a collection called `activity_with_population`, with the function from the set-up.

In [ ]:
combined = load_csv_to_collection("community_health_activity_with_population.csv", "activity_with_population")

Average waiting times (days) for 12 groups of patients. Guess the mean, minimum and maximum before you run the next cell.

In [ ]:
activity = pd.DataFrame(list(db["activity_with_population"].find({}, {"_id": 0})))
waits = activity["average_wait_days"]
print(waits.tolist())

print("Mean:              ", round(waits.mean(), 2))
print("Minimum:           ", waits.min())
print("Maximum:           ", waits.max())
print("Standard deviation:", round(waits.std(), 2))

Read it as: a typical wait is **8 days, give or take about 2**. The shortest is 5 days and the longest 12.

All at once with `.describe()`:

In [ ]:
waits.describe().round(2)

**Try it.** Suppose one group's wait was wrongly recorded as 120 days. The cell below makes a copy with that error. Which of the four measures changes most?

In [ ]:
wrong = waits.copy()
wrong.iloc[0] = 120

print("Mean:              ", round(wrong.mean(), 2))
print("Minimum:           ", wrong.min())
print("Maximum:           ", wrong.max())
print("Standard deviation:", round(wrong.std(), 2))

**Answer.** The maximum changes most, from 12 to 120. The standard deviation jumps from about 2 to over 32 and the mean rises from 8 to about 17. Only the minimum stays the same. One error distorts the mean and the spread, which is why the minimum and maximum are checked first.

### The same in MongoDB

When the data lives in MongoDB, it can do the summarising itself with `aggregate()`. It takes a list of **stages**; here there is one, `$group`.

- `"_id": None` means one group: every document.
- Each other line names a result and the operator that calculates it.
- `"$average_wait_days"`, with a `$` in front, means "the value of that field".

In [ ]:
pipeline = [
    {"$group": {
        "_id": None,
        "mean": {"$avg": "$average_wait_days"},
        "minimum": {"$min": "$average_wait_days"},
        "maximum": {"$max": "$average_wait_days"},
        "std_dev": {"$stdDevSamp": "$average_wait_days"},
    }}
]
for result in db["activity_with_population"].aggregate(pipeline):
    print(result)

The same numbers as pandas: `$stdDevSamp` and pandas `.std()` both calculate the **sample** standard deviation.

### Identifying variation between groups

Change `None` to `"$deprivation_band"` for one result per band. `$sort` puts the groups in order.

In [ ]:
pipeline = [
    {"$group": {
        "_id": "$deprivation_band",
        "mean": {"$avg": "$average_wait_days"},
        "minimum": {"$min": "$average_wait_days"},
        "maximum": {"$max": "$average_wait_days"},
        "std_dev": {"$stdDevSamp": "$average_wait_days"},
    }},
    {"$sort": {"mean": 1}},
]
pd.DataFrame(list(db["activity_with_population"].aggregate(pipeline))).round(2)

- The **means** differ: waits are longest in the high-deprivation area.
- The **ranges do not overlap**: every high-band wait (9–12) is longer than every low-band wait (5–7.5).
- The **standard deviations** are similar: each group is about as consistent as the others.

Each band here is a single area with four rows; real conclusions need many areas per group.

### Same average, same experience?

Group the waits by **year** instead.

In [ ]:
pipeline = [
    {"$group": {
        "_id": "$reporting_year",
        "rows": {"$sum": 1},
        "mean": {"$avg": "$average_wait_days"},
        "minimum": {"$min": "$average_wait_days"},
        "maximum": {"$max": "$average_wait_days"},
        "std_dev": {"$stdDevSamp": "$average_wait_days"},
    }},
    {"$sort": {"_id": 1}},
]
pd.DataFrame(list(combined.aggregate(pipeline))).round(2)

In [ ]:
for doc in combined.find({"reporting_year": 2025}, {"_id": 0, "area_name": 1, "age_group": 1, "average_wait_days": 1}):
    print(doc)

## 4. Join first, or load separately?

The combined file was made in Week 4 by joining two cleaned files, the appointments and the area populations, **before** anything went into MongoDB. The two files could also have gone in separately, as two collections.

In [ ]:
activity = load_csv_to_collection("community_health_activity_clean.csv", "activity")
population = load_csv_to_collection("area_population_clean.csv", "population")

The same fact, Northside's 2024 population, in each version:

In [ ]:
query = {"area_code": "CH001", "reporting_year": 2024}

print("In population:", population.count_documents({"area_code": "CH001", "year": 2024}), "document")
print("In the combined collection:", combined.count_documents(query), "documents")
for doc in combined.find(query, {"_id": 0, "age_group": 1, "population": 1}):
    print("   ", doc)

| | Two collections | One combined collection |
|---|---|---|
| Each fact is stored | once | on every matching row |
| A population figure changes | update one document | update every copy |
| A question that needs both | the two must be joined when you ask it | one query |
| Ready for the business question? | not yet: the join still has to be done | yes: every row already has what the question needs |

> **Question.** What are the pros and cons of joining the data **before** putting it into MongoDB, so that the descriptive analysis can be done in MongoDB?

**Answer.** 

> **Question.** A dashboard is read thousands of times a day, and the population figures change once a year. Two collections or one combined?

**Answer.**

---
# Part B — Your turn

## Task 1: Summarise and compare

**1a.** Bring the **Welsh** areas (`W06`) from the male collection into pandas. Find the mean, minimum, maximum and standard deviation of `AllAges`.
**1b.** Do the same for the **London boroughs** (`E09`).
**1c.** Which group has the larger mean? Which varies more (larger standard deviation)?

Expected: 1a mean 69,150.7, minimum 28,978, maximum 174,278, standard deviation 33,633.4; 1b mean 128,303.6, minimum 4,501, maximum 184,057, standard deviation 36,223.3.

In [ ]:
#your code here

## Task 2: Aggregation ladder

Each answer is one pipeline: a `$match` on the area code, then a `$group` with `"_id": None`.

| # | Question | Expected |
|---|---|---|
| 2a | Total females in Welsh areas (`W06`), and how many areas | 1,570,721 from 22 areas |
| 2b | Total males in Scottish council areas (`S12`), and how many areas | 2,596,530 from 32 areas |
| 2c | Average number of males over 61 (`>61`) in a London borough (`E09`) | 17,093.0 |
| 2d | Mean, minimum and maximum female `AllAges` in the London boroughs (`E09`) | mean 130,444.5, minimum 3,571, maximum 193,521 |

In [ ]:
#your code here

## Task 3: Summarise the combined data

**3a.** Which row has the **shortest** average wait, and which has the **longest**? Use `sort` and `limit(1)`, and show only `area_name`, `age_group` and `average_wait_days`.
**3b.** Use one `$group` to find the **mean, minimum and maximum** number of `appointments`.
**3c.** Use `$group` with `"_id": "$age_group"` to find the **mean wait for each age group**. Which age group waits longest?

Expected: 3a shortest Northside 18-64 (5.0 days), longest Hilltown 65+ (12.0 days); 3b mean 820, minimum 275, maximum 1,280; 3c 0-17 7.0, 18-64 7.5, 65+ 9.5 days.

In [ ]:
#your code here